In [1]:
# Heart disease


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score
)
 
DATA_PATH = "heart.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print(df.head())
print(df.info())

Shape: (920, 16)
   id  age     sex    dataset               cp  trestbps   chol    fbs  \
0   1   63    Male  Cleveland   typical angina     145.0  233.0   True   
1   2   67    Male  Cleveland     asymptomatic     160.0  286.0  False   
2   3   67    Male  Cleveland     asymptomatic     120.0  229.0  False   
3   4   37    Male  Cleveland      non-anginal     130.0  250.0  False   
4   5   41  Female  Cleveland  atypical angina     130.0  204.0  False   

          restecg  thalch  exang  oldpeak        slope   ca  \
0  lv hypertrophy   150.0  False      2.3  downsloping  0.0   
1  lv hypertrophy   108.0   True      1.5         flat  3.0   
2  lv hypertrophy   129.0   True      2.6         flat  2.0   
3          normal   187.0  False      3.5  downsloping  0.0   
4  lv hypertrophy   172.0  False      1.4    upsloping  0.0   

                thal  num  
0       fixed defect    0  
1             normal    2  
2  reversable defect    1  
3             normal    0  
4             norma

In [9]:
df = df.drop(columns=[c for c in ["id", "dataset"] if c in df.columns])

In [17]:
df = df.replace(["?", " ", ""], np.nan)

In [18]:
possible_target_names = ["target", "num", "HeartDisease", "heartdisease",
                          "output", "condition", "class", "diagnosis"]
target_col = next((c for c in possible_target_names if c in df.columns), None)
if target_col is None:
    raise ValueError(
        f"Could not find a target column. Your columns are: {df.columns.tolist()}\n"
        "Set target_col manually to whichever one is your label."
    )

In [19]:
feature_cols = [c for c in df.columns if c != target_col]
categorical_cols = df[feature_cols].select_dtypes(include="object").columns.tolist()
numeric_cols = [c for c in feature_cols if c not in categorical_cols]
 
print("\nCategorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)


Categorical columns: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']
Numeric columns: ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']


In [20]:
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

In [21]:
encoders = {}
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le 

C:\Users\Sharanabasava\AppData\Local\Temp\ipykernel_5356\41353058.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])
C:\Users\Sharanabasava\AppData\Local\Temp\ipykernel_5356\41353058.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])


In [22]:
if df[target_col].nunique() > 2:
    df["target"] = (pd.to_numeric(df[target_col], errors="coerce") > 0).astype(int)
    if target_col != "target":
        df = df.drop(columns=[target_col])
else:
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce").astype(int)
    if target_col != "target":
        df = df.rename(columns={target_col: "target"})

In [23]:
before = len(df)
df = df.drop_duplicates()
print(f"\nDropped {before - len(df)} duplicate rows")


Dropped 2 duplicate rows


In [24]:
print("\nRemaining missing values:\n", df.isnull().sum())
print("\nColumn types:\n", df.dtypes)
print("\nTarget balance:\n", df["target"].value_counts())


Remaining missing values:
 age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalch      0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

Column types:
 age           int64
sex           int64
cp            int64
trestbps    float64
chol        float64
fbs           int64
restecg       int64
thalch      float64
exang         int64
oldpeak     float64
slope         int64
ca          float64
thal          int64
target        int64
dtype: object

Target balance:
 target
1    508
0    410
Name: count, dtype: int64


In [26]:
plt.figure(figsize=(6, 4))
sns.countplot(x="target", data=df)
plt.title("Class balance: 0 = No Disease, 1 = Disease")
plt.savefig("class_balance.png", bbox_inches="tight")
plt.close()
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.savefig("correlation_heatmap.png", bbox_inches="tight")
plt.close()


In [27]:
X = df.drop(columns=["target"])
y = df["target"]

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [29]:
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)
 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [30]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
 
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train_scaled, y_train)
 
models = {"Logistic Regression": log_reg, "Decision Tree": tree}

In [31]:
results = {}
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results[name] = {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}
 
    print(f"\n--- {name} ---")
    print(f"Accuracy:  {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall:    {rec:.3f}")
    print(f"F1 Score:  {f1:.3f}")
    print(classification_report(y_test, y_pred))
 
results_df = pd.DataFrame(results).T
print("\nSummary:\n", results_df)
results_df.to_csv("model_results_summary.csv")


--- Logistic Regression ---
Accuracy:  0.793
Precision: 0.802
Recall:    0.833
F1 Score:  0.817
              precision    recall  f1-score   support

           0       0.78      0.74      0.76        82
           1       0.80      0.83      0.82       102

    accuracy                           0.79       184
   macro avg       0.79      0.79      0.79       184
weighted avg       0.79      0.79      0.79       184


--- Decision Tree ---
Accuracy:  0.821
Precision: 0.863
Recall:    0.804
F1 Score:  0.832
              precision    recall  f1-score   support

           0       0.78      0.84      0.81        82
           1       0.86      0.80      0.83       102

    accuracy                           0.82       184
   macro avg       0.82      0.82      0.82       184
weighted avg       0.82      0.82      0.82       184


Summary:
                      accuracy  precision    recall        f1
Logistic Regression  0.793478   0.801887  0.833333  0.817308
Decision Tree        0.82

In [32]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Disease", "Disease"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.savefig("confusion_matrices.png", bbox_inches="tight")
plt.close()

In [33]:
plt.figure(figsize=(7, 6))
for name, model in models.items():
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
 
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend(loc="lower right")
plt.savefig("roc_curves.png", bbox_inches="tight")
plt.close()


In [34]:
joblib.dump(log_reg, "logistic_regression_model.pkl")
joblib.dump(tree, "decision_tree_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(imputer, "imputer.pkl")
joblib.dump(encoders, "label_encoders.pkl")   # needed to encode new patients' text fields
joblib.dump(list(X_train.columns), "feature_order.pkl")
print("\nSaved models, scaler, imputer, encoders, and feature order.")


Saved models, scaler, imputer, encoders, and feature order.


In [38]:
def predict_patient(model, scaler, imputer, encoders, patient_dict, feature_order):
    x = pd.DataFrame([patient_dict])[feature_order].copy()
    for col, le in encoders.items():
        if col in x.columns:
            x[col] = le.transform(x[col].astype(str))
    x_imputed = imputer.transform(x)
    x_scaled = scaler.transform(x_imputed)
    pred = model.predict(x_scaled)[0]
    proba = model.predict_proba(x_scaled)[0][1]
    return pred, proba

In [39]:
print("\nValid categories per column (use these exact spellings below):")
for col, le in encoders.items():
    print(f"  {col} -> {list(le.classes_)}")


Valid categories per column (use these exact spellings below):
  sex -> ['Female', 'Male']
  cp -> ['asymptomatic', 'atypical angina', 'non-anginal', 'typical angina']
  fbs -> ['False', 'True']
  restecg -> ['lv hypertrophy', 'normal', 'st-t abnormality']
  exang -> ['False', 'True']
  slope -> ['downsloping', 'flat', 'upsloping']
  thal -> ['fixed defect', 'normal', 'reversable defect']


In [40]:
new_patient = {}
for col in X_train.columns:
    if col in encoders:
        mode_encoded_val = int(X_train[col].mode()[0])
        new_patient[col] = encoders[col].inverse_transform([mode_encoded_val])[0]
    else:
        new_patient[col] = X_train[col].mode()[0]
 
print("\nExample patient (auto-generated from most common values):")
print(new_patient)


Example patient (auto-generated from most common values):
{'age': np.int64(54), 'sex': 'Male', 'cp': 'asymptomatic', 'trestbps': np.float64(130.0), 'chol': np.float64(0.0), 'fbs': 'False', 'restecg': 'normal', 'thalch': np.float64(140.0), 'exang': 'False', 'oldpeak': np.float64(0.0), 'slope': 'flat', 'ca': np.float64(0.0), 'thal': 'normal'}


In [42]:
new_patient["age"] = 58
new_patient["chol"] = 289
pred, proba = predict_patient(log_reg, scaler, imputer, encoders, new_patient, X_train.columns)
print("\n--- New Patient Prediction (Logistic Regression) ---")
print(f"Result: {'Heart Disease Likely' if pred == 1 else 'No Heart Disease'}")
print(f"Probability of disease: {proba:.2%}")
 
pred_tree, proba_tree = predict_patient(tree, scaler, imputer, encoders, new_patient, X_train.columns)
print("\n--- New Patient Prediction (Decision Tree) ---")
print(f"Result: {'Heart Disease Likely' if pred_tree == 1 else 'No Heart Disease'}")
print(f"Probability of disease: {proba_tree:.2%}")

print("\nDone. Saved: class_balance.png, correlation_heatmap.png, "
      "confusion_matrices.png, roc_curves.png, model_results_summary.csv, "
      "logistic_regression_model.pkl, decision_tree_model.pkl, scaler.pkl, "
      "imputer.pkl, label_encoders.pkl")


--- New Patient Prediction (Logistic Regression) ---
Result: No Heart Disease
Probability of disease: 47.05%

--- New Patient Prediction (Decision Tree) ---
Result: No Heart Disease
Probability of disease: 38.81%

Done. Saved: class_balance.png, correlation_heatmap.png, confusion_matrices.png, roc_curves.png, model_results_summary.csv, logistic_regression_model.pkl, decision_tree_model.pkl, scaler.pkl, imputer.pkl, label_encoders.pkl
